# **Model Training Script**

## **Dependencies**

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import classification_report,confusion_matrix
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
import pickle
from freetranslate.googletranslate import GoogleTranslate
import re
from langdetect import detect, DetectorFactory

c:\Users\benij\ds_project\Ai_pettition_project\backend\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DetectorFactory.seed = 0

In [3]:
df=pd.read_csv(r"C:\Users\benij\ds_project\Ai_pettition_project\backend\data\petition_dataset.csv")

In [4]:
df['paragraph']=df['paragraph'].str.lower()

In [5]:
df['paragraph']

0      the residents of the eastside district have be...
1      the local historical society is calling for a ...
2      the intersection of miller road and 12th stree...
3      a political activist is currently being held i...
4      we are petitioning the parks and rec departmen...
                             ...                        
130    we are requesting that the municipal governmen...
131    a cruise ship in the [name] harbor has reporte...
132    a prominent intersection in the [name] suburb ...
133    we are petitioning for the creation of a 'loca...
134    a major wildfire has breached the perimeter of...
Name: paragraph, Length: 135, dtype: object

In [6]:
embedding_encoder=SentenceTransformer("all-mpnet-base-v2")

In [7]:
embeddings=embedding_encoder.encode(df['paragraph'].to_list(),show_progress_bar=True)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:13<00:00,  2.76s/it]


In [8]:
embeddings

array([[-0.03913742,  0.08554561,  0.00811043, ..., -0.00625137,
        -0.09654348,  0.01130023],
       [-0.00369807,  0.15785502, -0.0090647 , ...,  0.00048173,
        -0.00821634, -0.01667375],
       [-0.04810815,  0.06345277, -0.00410313, ..., -0.01186394,
         0.01526279, -0.01738363],
       ...,
       [-0.04106554,  0.03173032, -0.0114704 , ..., -0.00485279,
         0.02469925, -0.02962606],
       [-0.01453242,  0.11541045, -0.02053674, ..., -0.00827649,
        -0.00616393, -0.00649022],
       [-0.05948438,  0.04382149,  0.03417591, ...,  0.04384103,
        -0.09394819, -0.0060386 ]], dtype=float32)

In [9]:
embedding_df=pd.DataFrame(embeddings)

In [10]:
embedding_df

,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
0,-0.039137,0.085546,0.008110,0.005734,0.003092,-0.021437,0.030974,-0.004742,0.046045,-0.026761,...,-0.056044,-0.015120,-0.071530,0.037596,-0.028086,-0.051880,0.004215,-0.006251,-0.096543,0.011300
1,-0.003698,0.157855,-0.009065,0.057410,-0.049112,0.029283,0.019688,-0.010098,-0.039744,0.003240,...,-0.066848,-0.021047,-0.040657,-0.009027,-0.001248,-0.001137,-0.049499,0.000482,-0.008216,-0.016674
2,-0.048108,0.063453,-0.004103,0.059457,-0.007126,-0.010976,0.044030,-0.012931,-0.045540,-0.002470,...,0.035078,0.003521,-0.076478,0.003791,0.002145,-0.018664,-0.064494,-0.011864,0.015263,-0.017384
3,0.018013,0.076613,0.023235,-0.001097,-0.014360,0.004616,0.054259,-0.019657,0.060772,-0.004129,...,-0.033591,-0.022076,-0.029633,-0.026329,-0.006715,-0.056994,-0.006956,0.074214,-0.047328,0.037855
4,0.094151,0.173625,-0.003231,0.090680,-0.012524,-0.018788,0.001823,-0.013914,0.008168,-0.002371,...,-0.005451,0.078101,-0.047929,0.020364,0.030471,-0.037955,-0.007151,-0.034371,0.012097,0.032206
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,0.024388,0.088324,0.018484,0.052858,-0.002475,-0.026579,0.026617,0.000032,-0.054151,0.002900,...,-0.012149,0.013177,-0.060514,0.007044,0.012415,0.011397,0.002297,-0.017622,-0.017262,0.000856
131,-0.013549,0.056979,0.009368,-0.058381,-0.017121,-0.033527,0.012438,-0.005747,0.060768,0.000839,...,-0.072835,-0.060053,-0.031001,0.024623,0.001683,-0.009622,-0.006622,0.012329,-0.032355,-0.013222
132,-0.041066,0.031730,-0.011470,0.053356,-0.038801,-0.001520,0.041952,0.004940,-0.023711,-0.021729,...,0.002807,-0.015169,-0.059352,0.022217,0.013098,-0.071467,-0.066162,-0.004853,0.024699,-0.029626
133,-0.014532,0.115410,-0.020537,0.029462,-0.031436,0.000588,0.044283,0.010049,-0.042662,-0.000761,...,-0.019693,0.017628,-0.079624,0.003609,-0.022430,-0.015374,-0.062935,-0.008276,-0.006164,-0.006490


In [11]:
df

,paragraph,label
0,the residents of the eastside district have be...,Urgent Action Required
1,the local historical society is calling for a ...,Normal Action
2,the intersection of miller road and 12th stree...,Fast Action
3,a political activist is currently being held i...,Urgent Action Required
4,we are petitioning the parks and rec departmen...,Normal Action
...,...,...
130,we are requesting that the municipal governmen...,Normal Action
131,a cruise ship in the [name] harbor has reporte...,Urgent Action Required
132,a prominent intersection in the [name] suburb ...,Fast Action
133,we are petitioning for the creation of a 'loca...,Normal Action


In [12]:
le=LabelEncoder()

In [13]:
df['label']=le.fit_transform(df['label'])

In [14]:
df

,paragraph,label
0,the residents of the eastside district have be...,2
1,the local historical society is calling for a ...,1
2,the intersection of miller road and 12th stree...,0
3,a political activist is currently being held i...,2
4,we are petitioning the parks and rec departmen...,1
...,...,...
130,we are requesting that the municipal governmen...,1
131,a cruise ship in the [name] harbor has reporte...,2
132,a prominent intersection in the [name] suburb ...,0
133,we are petitioning for the creation of a 'loca...,1


In [15]:
X=embedding_df
y=df['label'].values

In [16]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto'],
    'class_weight': ['balanced'] 
}

grid_search = GridSearchCV(
    estimator=SVC(probability=True),
    param_grid=param_grid,
    cv=skf,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X, y)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.2f}")

final_model = grid_search.best_estimator_

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best Parameters: {'C': 1, 'class_weight': 'balanced', 'gamma': 'scale', 'kernel': 'linear'}
Best CV Accuracy: 0.93


In [17]:
final_model

,C,1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [18]:
le.inverse_transform(final_model.predict(embedding_encoder.encode(["A massive ice jam has formed on the [Name] River, causing water levels to rise three feet every hour and threatening to sweep away the support pillars of the historic downtown bridge. If the jam is not broken with controlled explosives within the next four hours, the resulting flash flood will inundate the lower business district and likely cause a total bridge collapse. We demand the immediate deployment of state demolition experts and emergency dive teams to clear the obstruction before the structural integrity of our city's main crossing is lost forever."])))

array([' Urgent Action Required'], dtype=object)

In [19]:
with open(r"C:\Users\benij\ds_project\Ai_pettition_project\backend\models\svc_model_v2.pickle","wb") as file:
    pickle.dump(final_model,file)

In [20]:
final_model.predict(embedding_df.head(20))

array([2, 1, 0, 2, 1, 0, 1, 0, 2, 1, 2, 1, 0, 0, 2, 1, 0, 0, 1, 0])

In [21]:
# confusion_matrix(df['label'].head(20),final_model.predict(embedding_df.head(20)))
print(classification_report(df['label'].tail(20),final_model.predict(embedding_df.tail(20))))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      1.00      1.00         6
           2       1.00      1.00      1.00         8

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20



In [22]:
final_model

,C,1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [23]:
text="[பெயர்] அணைக்கட்டில் இன்று காலை மிகப்பெரிய விரிசல் கண்டறியப்பட்டுள்ளது. அடுத்த 6 மணிநேரத்திற்குள் நீர் வெளியேற்றப்பட்டு சீரமைப்பு பணிகள் தொடங்கப்படாவிட்டால், அணையை ஒட்டியுள்ள 20 கிராமங்கள் வெள்ளத்தில் மூழ்கும் அபாயம் உள்ளது. உடனடியாக தேசிய பேரிடர் மீட்புப் படையினரை அனுப்பி, மக்களைப் பாதுகாப்பான இடங்களுக்கு மாற்ற வேண்டும். ஒவ்வொரு நிமிடமும் ஆயிரக்கணக்கான உயிர்கள் ஆபத்தில் உள்ளன"

In [24]:
for i in range(len(text)):
    print(i,text[i])

0 [
1 ப
2 ெ
3 ய
4 ர
5 ்
6 ]
7  
8 அ
9 ண
10 ை
11 க
12 ்
13 க
14 ட
15 ்
16 ட
17 ி
18 ல
19 ்
20  
21 இ
22 ன
23 ்
24 ற
25 ு
26  
27 க
28 ா
29 ல
30 ை
31  
32 ம
33 ி
34 க
35 ப
36 ்
37 ப
38 ெ
39 ர
40 ி
41 ய
42  
43 வ
44 ி
45 ர
46 ி
47 ச
48 ல
49 ்
50  
51 க
52 ண
53 ்
54 ட
55 ற
56 ி
57 ய
58 ப
59 ்
60 ப
61 ட
62 ்
63 ட
64 ு
65 ள
66 ்
67 ள
68 த
69 ு
70 .
71  
72 அ
73 ட
74 ு
75 த
76 ்
77 த
78  
79 6
80  
81 ம
82 ண
83 ி
84 ந
85 ே
86 ர
87 த
88 ்
89 த
90 ி
91 ற
92 ்
93 க
94 ு
95 ள
96 ்
97  
98 ந
99 ீ
100 ர
101 ்
102  
103 வ
104 ெ
105 ள
106 ி
107 ய
108 ே
109 ற
110 ்
111 ற
112 ப
113 ்
114 ப
115 ட
116 ்
117 ட
118 ு
119  
120 ச
121 ீ
122 ர
123 ம
124 ை
125 ப
126 ்
127 ப
128 ு
129  
130 ப
131 ண
132 ி
133 க
134 ள
135 ்
136  
137 த
138 ொ
139 ட
140 ங
141 ்
142 க
143 ப
144 ்
145 ப
146 ட
147 ா
148 வ
149 ி
150 ட
151 ்
152 ட
153 ா
154 ல
155 ்
156 ,
157  
158 அ
159 ண
160 ை
161 ய
162 ை
163  
164 ஒ
165 ட
166 ்
167 ட
168 ி
169 ய
170 ு
171 ள
172 ்
173 ள
174  
175 2
176 0
177  
178 க
179 ி
180 ர
181 ா
182 ம
183 ங
184 ்


In [25]:
sentences = re.split(r'[.!?]+', text)
sentences = [s.strip() for s in sentences if s.strip()]

print(sentences)


['[பெயர்] அணைக்கட்டில் இன்று காலை மிகப்பெரிய விரிசல் கண்டறியப்பட்டுள்ளது', 'அடுத்த 6 மணிநேரத்திற்குள் நீர் வெளியேற்றப்பட்டு சீரமைப்பு பணிகள் தொடங்கப்படாவிட்டால், அணையை ஒட்டியுள்ள 20 கிராமங்கள் வெள்ளத்தில் மூழ்கும் அபாயம் உள்ளது', 'உடனடியாக தேசிய பேரிடர் மீட்புப் படையினரை அனுப்பி, மக்களைப் பாதுகாப்பான இடங்களுக்கு மாற்ற வேண்டும்', 'ஒவ்வொரு நிமிடமும் ஆயிரக்கணக்கான உயிர்கள் ஆபத்தில் உள்ளன']


In [26]:
g = GoogleTranslate()

result = await g.translate(sentences[0], "en")
print(result.translated_text)


A massive crack was discovered this morning in the [name] dam


In [27]:
en_text=""

In [28]:
for i in range(len(sentences)):
    result = await g.translate(sentences[i], "en")
    en_text+=result.translated_text

In [29]:
en_text

'A massive crack was discovered this morning in the [name] damAs many as 20 villages adjacent to the dam are at risk of inundation if the water is not released and repair work started within the next 6 hours.Immediately dispatch the National Disaster Response Force and evacuate people to safer placesThousands of lives are at risk every minute'

In [30]:
le.inverse_transform(final_model.predict(embedding_encoder.encode([en_text])))

array([' Urgent Action Required'], dtype=object)

In [31]:
result = await g.translate("मेरा नाम माइक है", "en")
print(result.translated_text)



my name is Mike


In [32]:
text = "[பெயர்] அணைக்கட்டில் இன்று காலை"
lang = detect(text)

print(lang)


ta


In [33]:
df['paragraph']

0      the residents of the eastside district have be...
1      the local historical society is calling for a ...
2      the intersection of miller road and 12th stree...
3      a political activist is currently being held i...
4      we are petitioning the parks and rec departmen...
                             ...                        
130    we are requesting that the municipal governmen...
131    a cruise ship in the [name] harbor has reporte...
132    a prominent intersection in the [name] suburb ...
133    we are petitioning for the creation of a 'loca...
134    a major wildfire has breached the perimeter of...
Name: paragraph, Length: 135, dtype: object

In [34]:
final_model

,C,1
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [35]:
with open(r"C:\Users\benij\ds_project\Ai_pettition_project\backend\models\svc_model_v2.pickle","rb")as file:
    load_model=pickle.load(file)



In [36]:
type(load_model)

sklearn.svm._classes.SVC

In [37]:
le.classes_

array([' Fast Action', ' Normal Action', ' Urgent Action Required'],
      dtype=object)

In [40]:
with open(r"C:\Users\benij\ds_project\Ai_pettition_project\backend\encoder\label_encoder.pickle","wb") as f:
    pickle.dump(le,f)